# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/russelfordguinoo-oss/Russel-Repository-/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Baseline rule

Prioritize content that has not been updated for 91–180 days and has at least 300 search impressions in the last 90 days. Rank qualifying content by impressions so higher-visibility pages are reviewed first.

Signal verdicts:
- Staleness: MIXED — decline rates rise through 91–180 days but fall in the 181+ bucket.
- Impressions: MIXED — moderate and good impression groups have higher decline rates, but excellent-volume pages have a lower rate.

Reason code:
- stale_visible

Action:
- refresh_page

In [70]:
!git clone https://github.com/russelfordguinoo-oss/Russel-Repository-.git

fatal: destination path 'Russel-Repository-' already exists and is not an empty directory.


In [71]:
%cd /content/Russel-Repository-

/content/Russel-Repository-


In [72]:
!ls

01_first_look_and_discovery.ipynb   GUIDE.md		scripts
02_your_first_readable_model.ipynb  LICENSE		SETUP.md
AGENTS.md			    notebooks		skills
CLAUDE.md			    outputs		submission
data				    README.md		w03_data_contract.ipynb
DATA_USE.md			    requirements.txt	work
docs				    Russel-Repository-


In [73]:
!ls data/raw

content_refresh_anonymized.csv


In [74]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [75]:
signals = [
    "days_since_last_update",
    "impressions_90d"
]

df[signals].describe()

,days_since_last_update,impressions_90d
count,30000.000000,30000.000000
mean,46.098300,5200.366300
std,42.078709,16838.019547
min,1.000000,1.000000
25%,20.000000,81.000000
50%,20.000000,731.000000
75%,104.000000,3615.250000
max,373.000000,517715.000000


In [76]:
df[signals].isna().sum()

,0
days_since_last_update,0
impressions_90d,0


In [77]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [78]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30", "31-90", "91-180", "181+"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("is_declining_label", "size"),
          decline_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

staleness_check["decline_rate"] = (
    staleness_check["decline_rate"] * 100
).round(1)

staleness_check

,staleness_bucket,n,decline_rate
0,0-30,20480,51.1
1,31-90,175,58.9
2,91-180,9171,61.1
3,181+,174,47.1


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [79]:
# Define the two conditions from our baseline rule
stale = (
    df["days_since_last_update"].between(91, 180)
).astype(int)

visible = (
    df["impressions_90d"] >= 300
).astype(int)

# Transparent baseline score
df["score"] = (
    stale
    * visible
    * df["impressions_90d"]
)

# Reason code
df["reason_code"] = ""

df.loc[
    (stale == 1) & (visible == 1),
    "reason_code"
] = "stale_visible"

# Action label
df["action"] = ""

df.loc[
    (stale == 1) & (visible == 1),
    "action"
] = "refresh_page"

# Rank highest score first
ranked = df.sort_values(
    "score",
    ascending=False
).copy()

# Keep the useful columns for the ranked queue
queue = ranked[
    [
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d"
    ]
]

# Write the required CSV
import os

os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Queue written successfully.")
print("Rows:", len(queue))

queue.head(10)

Queue written successfully.
Rows: 30000


,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d
6653,content_5fe46e04994d,client_4e07408562,517715,stale_visible,refresh_page,104,517715
29400,content_2dba2b1f9536,client_6208ef0f77,443434,stale_visible,refresh_page,104,443434
13537,content_2c2606c5d176,client_19581e27de,347399,stale_visible,refresh_page,104,347399
26531,content_cb112fce36be,client_19581e27de,309910,stale_visible,refresh_page,104,309910
21565,content_9532f197bbc8,client_4e07408562,309192,stale_visible,refresh_page,104,309192
3394,content_36ff89c8214e,client_19581e27de,295097,stale_visible,refresh_page,104,295097
26798,content_b28d1efd668f,client_6208ef0f77,286608,stale_visible,refresh_page,104,286608
23767,content_813e88069237,client_6208ef0f77,233561,stale_visible,refresh_page,104,233561
26255,content_c21024970297,client_19581e27de,211366,stale_visible,refresh_page,104,211366
7445,content_c8e9d6ab9013,client_19581e27de,208678,stale_visible,refresh_page,104,208678


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review

The baseline recommends refresh_page for all top-20 rows because they are 91–180 days since their last update and have at least 300 impressions. Confidence varies because the rule does not consider CTR or average position when ranking. A recommendation could be wrong if the page is already performing strongly, especially at a high search position.

In [80]:
review_notes = {
    1: ("High", "The page has very high impressions, a strong average position of 4.2, and very low CTR. It could be a strong refresh candidate, but changing a high-ranking page could still hurt performance."),
    2: ("Medium", "The page has very high impressions, but its average position is 27.9. Low CTR may mainly reflect weak ranking rather than stale content."),
    3: ("Medium", "The page has high impressions and position 4.2, but its CTR of 0.53% is not as clearly weak. A refresh may not produce much improvement."),
    4: ("High", "The page has high impressions, position 5.6, and low CTR of 0.16%, making it a plausible refresh candidate."),
    5: ("Low", "The page already averages position 2.0 and has a relatively higher CTR of 0.87%. Refreshing successful content could introduce unnecessary risk."),
    6: ("High", "The page has high impressions, position 7.3, and extremely low CTR of 0.05%, giving a strong reason to investigate it."),
    7: ("Medium", "The page has high impressions but averages position 26.2. Low CTR may be explained by its ranking rather than staleness."),
    8: ("Medium", "The page has high impressions but position 26.2. Improving ranking may matter more than refreshing the content."),
    9: ("Medium", "The page has strong visibility and position 5.1, but CTR of 0.41% does not by itself prove that the content needs refreshing."),
    10: ("High", "The page has high impressions, position 9.7, and a CTR of 0.00%, making it a strong candidate for investigation."),
    11: ("Medium", "The page has high impressions but position 27.9, so low CTR may be primarily caused by weak search ranking."),
    12: ("Medium", "The page has strong visibility and position 5.8, but CTR of 0.24% provides only moderate evidence for a refresh."),
    13: ("Medium", "The page has strong visibility and position 5.7, but CTR of 0.24% does not clearly establish that stale content is the problem."),
    14: ("Medium", "The page has substantial impressions and position 12.5. Its CTR of 0.24% could reflect ranking limitations rather than content freshness."),
    15: ("Medium", "The page has high impressions and position 4.3, but CTR of 0.24% gives only moderate evidence that refreshing would help."),
    16: ("Medium", "The page has high impressions and position 4.0, but its CTR of 0.45% is not extremely low. A refresh may not be necessary."),
    17: ("High", "The page has high impressions, position 6.2, and low CTR of 0.16%, providing a reasonable case for investigation."),
    18: ("Medium", "The page has high impressions but position 25.8. Low CTR may mainly reflect its ranking rather than stale content."),
    19: ("Medium", "The page has high impressions but position 22.1. Improving search position may be more appropriate than refreshing content."),
    20: ("Medium", "The page has high impressions and position 4.3, but CTR of 0.23% provides only moderate evidence for a refresh.")
}

for rank, (confidence, wrong_reason) in review_notes.items():
    review.loc[review["rank"] == rank, "confidence"] = confidence
    review.loc[review["rank"] == rank, "what_would_make_it_wrong"] = wrong_reason

review[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "confidence",
        "what_would_make_it_wrong"
    ]
]

,rank,content_id,action,reason_code,confidence,what_would_make_it_wrong
6653,1,content_5fe46e04994d,refresh_page,stale_visible,High,"The page has very high impressions, a strong a..."
29400,2,content_2dba2b1f9536,refresh_page,stale_visible,Medium,"The page has very high impressions, but its av..."
13537,3,content_2c2606c5d176,refresh_page,stale_visible,Medium,The page has high impressions and position 4.2...
26531,4,content_cb112fce36be,refresh_page,stale_visible,High,"The page has high impressions, position 5.6, a..."
21565,5,content_9532f197bbc8,refresh_page,stale_visible,Low,The page already averages position 2.0 and has...
3394,6,content_36ff89c8214e,refresh_page,stale_visible,High,"The page has high impressions, position 7.3, a..."
26798,7,content_b28d1efd668f,refresh_page,stale_visible,Medium,The page has high impressions but averages pos...
23767,8,content_813e88069237,refresh_page,stale_visible,Medium,The page has high impressions but position 26....
26255,9,content_c21024970297,refresh_page,stale_visible,Medium,The page has strong visibility and position 5....
7445,10,content_c8e9d6ab9013,refresh_page,stale_visible,High,"The page has high impressions, position 9.7, a..."


In [81]:
review = ranked.head(20)[
    [
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "avg_position"
    ]
].copy()

review.insert(0, "rank", range(1, len(review) + 1))

review["confidence"] = ""
review["what_would_make_it_wrong"] = ""

review

,rank,content_id,score,reason_code,action,days_since_last_update,impressions_90d,ctr,avg_position,confidence,what_would_make_it_wrong
6653,1,content_5fe46e04994d,517715,stale_visible,refresh_page,104,517715,0.14,4.2,,
29400,2,content_2dba2b1f9536,443434,stale_visible,refresh_page,104,443434,0.21,27.9,,
13537,3,content_2c2606c5d176,347399,stale_visible,refresh_page,104,347399,0.53,4.2,,
26531,4,content_cb112fce36be,309910,stale_visible,refresh_page,104,309910,0.16,5.6,,
21565,5,content_9532f197bbc8,309192,stale_visible,refresh_page,104,309192,0.87,2.0,,
3394,6,content_36ff89c8214e,295097,stale_visible,refresh_page,104,295097,0.05,7.3,,
26798,7,content_b28d1efd668f,286608,stale_visible,refresh_page,104,286608,0.06,26.2,,
23767,8,content_813e88069237,233561,stale_visible,refresh_page,104,233561,0.06,26.2,,
26255,9,content_c21024970297,211366,stale_visible,refresh_page,104,211366,0.41,5.1,,
7445,10,content_c8e9d6ab9013,208678,stale_visible,refresh_page,104,208678,0.00,9.7,,


In [82]:
review_data = ranked.head(20)[
    [
        "content_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "avg_position"
    ]
].copy()

print(review_data.to_string(index=False))

          content_id  score   reason_code       action  days_since_last_update  impressions_90d  ctr  avg_position
content_5fe46e04994d 517715 stale_visible refresh_page                     104           517715 0.14           4.2
content_2dba2b1f9536 443434 stale_visible refresh_page                     104           443434 0.21          27.9
content_2c2606c5d176 347399 stale_visible refresh_page                     104           347399 0.53           4.2
content_cb112fce36be 309910 stale_visible refresh_page                     104           309910 0.16           5.6
content_9532f197bbc8 309192 stale_visible refresh_page                     104           309192 0.87           2.0
content_36ff89c8214e 295097 stale_visible refresh_page                     104           295097 0.05           7.3
content_b28d1efd668f 286608 stale_visible refresh_page                     104           286608 0.06          26.2
content_813e88069237 233561 stale_visible refresh_page                     104  

In [83]:
top20 = ranked.head(20)[
    [
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "avg_position",
        "content_age_days"
    ]
]

top20

,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,ctr,avg_position,content_age_days
6653,content_5fe46e04994d,client_4e07408562,517715,stale_visible,refresh_page,104,517715,0.14,4.2,537
29400,content_2dba2b1f9536,client_6208ef0f77,443434,stale_visible,refresh_page,104,443434,0.21,27.9,299
13537,content_2c2606c5d176,client_19581e27de,347399,stale_visible,refresh_page,104,347399,0.53,4.2,362
26531,content_cb112fce36be,client_19581e27de,309910,stale_visible,refresh_page,104,309910,0.16,5.6,126
21565,content_9532f197bbc8,client_4e07408562,309192,stale_visible,refresh_page,104,309192,0.87,2.0,445
3394,content_36ff89c8214e,client_19581e27de,295097,stale_visible,refresh_page,104,295097,0.05,7.3,144
26798,content_b28d1efd668f,client_6208ef0f77,286608,stale_visible,refresh_page,104,286608,0.06,26.2,153
23767,content_813e88069237,client_6208ef0f77,233561,stale_visible,refresh_page,104,233561,0.06,26.2,153
26255,content_c21024970297,client_19581e27de,211366,stale_visible,refresh_page,104,211366,0.41,5.1,126
7445,content_c8e9d6ab9013,client_19581e27de,208678,stale_visible,refresh_page,104,208678,0.00,9.7,362


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks

- Rank 5 is a weak pick because it already averages position 2.0 with a 0.87% CTR. Refreshing a strong-ranking page may introduce unnecessary risk.
- Rank 2 is a weak pick because its average position is 27.9. Its low CTR may be explained by weak ranking rather than stale content.
- Ranks 7, 8, 11, 18, and 19 have similarly deep average positions, so low CTR could reflect ranking rather than content freshness.

Leakage check

The baseline score uses only days_since_last_update and impressions_90d. It does not use trend_direction, trend_pct, or is_declining_label. These label-derived fields are used only for signal auditing and are not inputs to the score.

In [84]:
baseline_features = [
    "days_since_last_update",
    "impressions_90d"
]

leakage_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

print("Baseline features:")
print(baseline_features)

print("\nLeakage columns:")
print(leakage_columns)

print("\nLeakage columns used in score:")
print(
    [col for col in leakage_columns if col in baseline_features]
)

Baseline features:
['days_since_last_update', 'impressions_90d']

Leakage columns:
['trend_direction', 'trend_pct', 'is_declining_label']

Leakage columns used in score:
[]


In [85]:
saved_queue = pd.read_csv(
    "work/outputs/baseline_action_score.csv"
)

print("Saved rows:", len(saved_queue))
print("Saved columns:")
print(saved_queue.columns.tolist())

print("\nTop 5 saved rows:")
saved_queue.head()

Saved rows: 30000
Saved columns:
['content_id', 'client_id', 'score', 'reason_code', 'action', 'days_since_last_update', 'impressions_90d']

Top 5 saved rows:


,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d
0,content_5fe46e04994d,client_4e07408562,517715,stale_visible,refresh_page,104,517715
1,content_2dba2b1f9536,client_6208ef0f77,443434,stale_visible,refresh_page,104,443434
2,content_2c2606c5d176,client_19581e27de,347399,stale_visible,refresh_page,104,347399
3,content_cb112fce36be,client_19581e27de,309910,stale_visible,refresh_page,104,309910
4,content_9532f197bbc8,client_4e07408562,309192,stale_visible,refresh_page,104,309192


Self-check

- Two signal checks are included with visible bucket tables and n: staleness and impressions.
- Both signal verdicts are MIXED and are supported by the observed bucket results.
- Staleness is linked to the FlyRank refresh/staleness logic.
- The baseline rule is written in plain language before the scoring code.
- The score uses only days_since_last_update and impressions_90d.
- The reason code is stale_visible.
- The action label is refresh_page.
- The ranked queue contains 30,000 rows and is written to work/outputs/baseline_action_score.csv.
- The Top-20 review includes an action, reason code, confidence, and what would make the recommendation wrong.
- Weak picks were identified without changing the frozen baseline.
- trend_direction, trend_pct, and is_declining_label are not used as baseline scoring inputs.
- No future-window or label-derived inputs are used in the baseline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.